<a href="https://colab.research.google.com/github/jaspreet-aidev/RiceDoctor-EdgeAI/blob/main/training_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [3]:
# 1. Download the Rice Disease Dataset (Example: 19k cleaned dataset)
!kaggle datasets download -d chaitanyakamble69/rice-leaf-disease-riceguard-19k-cleaned

# 2. Download the Background/Negative Class Dataset (Random natural images)
!kaggle datasets download -d prasunroy/natural-images

# 3. Unzip them silently (-q means quiet, so it doesn't crash the browser with text)
!unzip -q rice-leaf-disease-riceguard-19k-cleaned.zip -d raw_rice_data/
!unzip -q natural-images.zip -d raw_background_data/


Dataset URL: https://www.kaggle.com/datasets/chaitanyakamble69/rice-leaf-disease-riceguard-19k-cleaned
License(s): MIT
100% 1.95G/1.95G [00:19<00:00, 107MB/s]

Dataset URL: https://www.kaggle.com/datasets/prasunroy/natural-images
License(s): CC-BY-NC-SA-4.0
100% 342M/342M [00:02<00:00, 155MB/s]



In [4]:
import os
import shutil
import random

# Create the final Master Dataset folder
base_dir = 'Master_Dataset'
os.makedirs(base_dir, exist_ok=True)

# 1. Move your rice folders into the Master Dataset
# (Adjust 'raw_rice_data' based on the exact unzipped folder name)
shutil.move('raw_rice_data', f'{base_dir}/Rice_Leaves')

# 2. Create the Background_Other folder
bg_dir = f'{base_dir}/Background_Other'
os.makedirs(bg_dir, exist_ok=True)

# 3. Pull 1000 random images from the natural images dataset to act as "Noise"
source_bg = 'raw_background_data/natural_images'
all_bg_images = []
for root, dirs, files in os.walk(source_bg):
    for file in files:
        if file.endswith(('jpg', 'jpeg', 'png')):
            all_bg_images.append(os.path.join(root, file))

# Randomly select 1000 images and copy them to your Background folder
selected_bg = random.sample(all_bg_images, 1000)
for img in selected_bg:
    shutil.copy(img, bg_dir)

print("Data Architecture Complete. Ready for Model Injection.")

Data Architecture Complete. Ready for Model Injection.


In [7]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Initialize the Data Generator with aggressive augmentation
datagen = ImageDataGenerator(
    rescale=1./255,          # Crush math values to 0-1
    rotation_range=40,       # Simulate bad camera angles
    width_shift_range=0.2,
    height_shift_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    validation_split=0.2     # Keep 20% of data for testing/validation
)

# Load the Training Data
train_generator = datagen.flow_from_directory(
    'Master_Dataset',
    target_size=(224, 224),  # MobileNetV2 strict requirement
    batch_size=32,
    class_mode='categorical',
    subset='training'
)

# Load the Validation Data
val_generator = datagen.flow_from_directory(
    'Master_Dataset',
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='validation'
)

Found 15647 images belonging to 2 classes.
Found 3911 images belonging to 2 classes.


In [6]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras import regularizers
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# 1. Define image dimensions and batch size
IMG_SIZE = 224
BATCH_SIZE = 32

# 3. Load Pre-trained MobileNetV2 Base (Freezing the Feature Extractor)
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))
base_model.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

checkpoint = ModelCheckpoint(
    '/content/drive/MyDrive/RiceDoctor_MobileNetV2.h5',
    monitor='val_accuracy',
    save_best_only=True,
    verbose=1
)

# 7. Execute Training Loop
history = model.fit(
    train_generator,
    epochs=25,
    validation_data=val_generator,
    callbacks=[early_stopping, checkpoint]
)

print("Training Complete. Best model saved permanently to Google Drive.")